In [1]:
# 1. 한국어 폰트(나눔고딕) 및 필요 라이브러리 설치 (최초 1회 약 1~2분 소요)
!apt-get update -qq
!apt-get install fonts-nanum* -qq
!wget -q http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bzip2 -dk shape_predictor_68_face_landmarks.dat.bz2
!pip install -q dlib opencv-python-headless Pillow

import cv2
import dlib
import numpy as np
from google.colab import files
from PIL import ImageFont, ImageDraw, Image

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package fonts-nanum.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Selecting previously unselected package fonts-nanum-coding.
Preparing to unpack .../fonts-nanum-coding_2.5-3_all.deb ...
Unpacking fonts-nanum-coding (2.5-3) ...
Selecting previously unselected package fonts-nanum-eco.
Preparing to unpack .../fonts-nanum-eco_1.000-7_all.deb ...
Unpacking fonts-nanum-eco (1.000-7) ...
Selecting previously unselected package fonts-nanum-extra.
Preparing to unpack .../fonts-nanum-extra_20200506-1_all.deb ...
Unpacking fonts-nanum-extra (20200506-1) ...
Setting up fonts-nanum-extra (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Setting up fo

In [2]:

# 2. 분석할 영상 업로드
print("면접 영상 파일을 업로드해주세요 (.mp4, .avi 등)")
uploaded = files.upload()
input_video_name = list(uploaded.keys())[0]
output_video_name = 'interview_feedback_' + input_video_name

# 3. Dlib 및 3D 모델 세팅
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

face3d = np.array([
    (0.0, 0.0, 0.0),            # 코 끝
    (0.0, -330.0, -65.0),       # 턱
    (-225.0, 170.0, -135.0),    # 왼쪽 눈 바깥
    (225.0, 170.0, -135.0),     # 오른쪽 눈 바깥
    (-150.0, -150.0, -125.0),   # 왼쪽 입꼬리
    (150.0, -150.0, -125.0)     # 오른쪽 입꼬리
], dtype=np.float64)

# 4. 영상 처리 준비
cap = cv2.VideoCapture(input_video_name)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_name, fourcc, fps, (width, height))

# 폰트 세팅 (설치된 나눔고딕 폰트 경로 지정, 폰트 크기 40)
fontpath = "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf"
font = ImageFont.truetype(fontpath, 40)

print(f"\n'{input_video_name}' 면접 자세 분석 시작...")

frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    frame_count += 1
    if frame_count % 30 == 0: print(f"{frame_count} 프레임 처리 완료")

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = detector(gray)

    # OpenCV 이미지를 한글 작성을 위해 PIL 이미지로 변환
    frame_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(frame_pil)

    for face in faces:
        landmarks = predictor(gray, face)

        face2d = np.array([
            (landmarks.part(30).x, landmarks.part(30).y),
            (landmarks.part(8).x, landmarks.part(8).y),
            (landmarks.part(36).x, landmarks.part(36).y),
            (landmarks.part(45).x, landmarks.part(45).y),
            (landmarks.part(48).x, landmarks.part(48).y),
            (landmarks.part(54).x, landmarks.part(54).y)
        ], dtype=np.float64)

        focal_length = 1 * width
        cam_matrix = np.array([[focal_length, 0, height/2],
                               [0, focal_length, width/2],
                               [0, 0, 1]], dtype=np.float64)

        success, rot_vec, trans_vec = cv2.solvePnP(face3d, face2d, cam_matrix, np.zeros((4, 1), dtype=np.float64))
        rmat, _ = cv2.Rodrigues(rot_vec)
        angles, _, _, _, _, _ = cv2.RQDecomp3x3(rmat)

        # 각도 계산 (Z값이 Roll)
        x = angles[0] # Pitch (끄덕임)
        y = angles[1] # Yaw (도리도리)
        z = angles[2] # Roll (갸우뚱)

        # 왼쪽 상단에 현재 Roll 수치 출력 (디버깅/확인용)
        draw.text((20, 20), f"현재 고개 기울기 (Roll): {int(z)}도", font=ImageFont.truetype(fontpath, 30), fill=(255, 255, 255))

        # 🚨 [핵심 피드백 로직] Roll 각도의 절댓값이 10도를 넘어가면 경고 문구 출력
        if abs(z) > 10:
            # 텍스트 내용, 위치, 색상(빨간색 RGB) 지정
            feedback_text = " 고개가 기울어져 있습니다. 고개를 세워주세요."
            draw.text((width//2 - 400, height - 100), feedback_text, font=font, fill=(255, 0, 0))

    # PIL 이미지를 다시 OpenCV용으로 변환
    frame = cv2.cvtColor(np.array(frame_pil), cv2.COLOR_RGB2BGR)

    out.write(frame)

cap.release()
out.release()
print(f"\n✅ 분석 완료! 파일명: {output_video_name}")
files.download(output_video_name)

면접 영상 파일을 업로드해주세요 (.mp4, .avi 등)


Saving bad_test.mp4 to bad_test.mp4

'bad_test.mp4' 면접 자세 분석 시작...
30 프레임 처리 완료
60 프레임 처리 완료
90 프레임 처리 완료
120 프레임 처리 완료
150 프레임 처리 완료
180 프레임 처리 완료
210 프레임 처리 완료
240 프레임 처리 완료
270 프레임 처리 완료
300 프레임 처리 완료
330 프레임 처리 완료
360 프레임 처리 완료
390 프레임 처리 완료
420 프레임 처리 완료
450 프레임 처리 완료
480 프레임 처리 완료
510 프레임 처리 완료
540 프레임 처리 완료
570 프레임 처리 완료
600 프레임 처리 완료

✅ 분석 완료! 파일명: interview_feedback_bad_test.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>